<a href="https://colab.research.google.com/github/summiyazeb/WEB-SEARCH-ENGINE/blob/main/Notebooks/02-Lexicon-and-Indexes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#
# Cell 1A: mount Google Drive and define file paths
#
import os
import json
import time
from google.colab import drive

# Connect Colab to Google Drive
drive.mount('/content/drive')

# Folder paths
BASE_DIR = '/content/drive/MyDrive/DSA_Search_Engine' #the project folder.
INDEX_DIR = os.path.join(BASE_DIR, 'index_data')   # all index outputs go here

# --- Input / output file paths ---
DATASET_PATH = os.path.join(BASE_DIR, 'cleaned_dataset.jsonl')   # full 55,000 reviews
SAMPLE_PATH  = os.path.join(BASE_DIR, 'sample_dataset.jsonl')    # 1,000-review test file

# Create index_data/ if it doesn't exist yet
os.makedirs(INDEX_DIR, exist_ok=True)

#  Quick check that everything is where we expect
print("Dataset found :", os.path.exists(DATASET_PATH))
print("index_data/   :", os.path.exists(INDEX_DIR))

Mounted at /content/drive
Dataset found : True
index_data/   : True


In [6]:
#
# Cell 1B: Tokenizer : cuts a piece of text into a clean list of words that the indexes can use.
# LOWERCASE each word
# trims extra appostrophies
#Stemming: coverts words to rooot words, eg drifting to drift

import re # regex: we use ot to find words
from functools import lru_cache # its a memoery, once we have found smth, it will remeber it, reduces time
import nltk # human lang library for stemming and stopwords
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer # rules for stemming, words-> roots

nltk.download('stopwords', quiet=True) # to download it and quiet if ale exists

STOPWORDS = set(stopwords.words('english')) # returns list of eng stopwords
_stemmer = PorterStemmer()

@lru_cache(maxsize=None)          # each unique word is stemmed only once
def stem_word(word):              # helps lru_cache to remember words
    return _stemmer.stem(word)

WORD_PATTERN = re.compile(r"[a-z0-9']+")   # a run of letters/digits/apostrophes

def tokenize(text, remove_stopwords=False):
    """Return a list of stemmed lowercase tokens.
    Stopwords are KEPT by default so 'data structures and algorithms'
    can match on 'and' (spec Requirement 5)."""
    tokens = []
    for w in WORD_PATTERN.findall(text.lower()):
        w = w.strip("'")
        if not w:
            continue
        if remove_stopwords and w in STOPWORDS:
            continue
        tokens.append(stem_word(w))
    return tokens

print(tokenize("The battery life is GREAT, but the controller drifts!"))

['the', 'batteri', 'life', 'is', 'great', 'but', 'the', 'control', 'drift']


In [7]:
#
# Cell 1C: Create a 1,000-review sample file for testing
#
SAMPLE_SIZE = 1000

count = 0
# Read the big file one line at a time; write only the first 1,000 lines
# r means read, utf-8 is for handling special characters
# w means write
with open(DATASET_PATH, 'r', encoding='utf-8') as src, \
     open(SAMPLE_PATH, 'w', encoding='utf-8') as dst:

    for line in src:  # to write line by line from src to dst
        dst.write(line)
        count += 1
        if count >= SAMPLE_SIZE:
            break

print(f"Sample created with {count} reviews") # count no of reviews stored
print(f"Saved at: {SAMPLE_PATH}")

#  Check: read it back, count the lines, inspect one timestamp ---
with open(SAMPLE_PATH, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

print(f"Lines in sample file: {len(records)}")
print(f"First doc_id: {records[0]['doc_id']} | Last doc_id: {records[-1]['doc_id']}")
print(f"Example timestamp: {records[0]['timestamp']}")

Sample created with 1000 reviews
Saved at: /content/drive/MyDrive/DSA_Search_Engine/sample_dataset.jsonl
Lines in sample file: 1000
First doc_id: 0 | Last doc_id: 999
Example timestamp: 1602718512453


In [8]:
# for the extra dataset (handling ,0)
# Cell 2A: Settings + normalize_record (safe defaults)
#
USE_SAMPLE = True      # True = 1,000-review test run, False = full 55,000

SOURCE_PATH = SAMPLE_PATH if USE_SAMPLE else DATASET_PATH
OUT_DIR = os.path.join(INDEX_DIR, 'sample') if USE_SAMPLE else INDEX_DIR
os.makedirs(OUT_DIR, exist_ok=True)

LEXICON_PATH  = os.path.join(OUT_DIR, 'lexicon.json')
FORWARD_PATH  = os.path.join(OUT_DIR, 'forward_index.jsonl')
INVERTED_PATH = os.path.join(OUT_DIR, 'inverted_index.json')
META_PATH     = os.path.join(OUT_DIR, 'index_meta.json')

DEFAULT_RATING = 3.0   # neutral value: 0 would sit below the real 1-5 range


def _to_number(value, cast, default):
    """Convert value using cast (int or float).
    Returns default if value is None or cannot be converted."""
    if value is None:
        return default
    try:
        return cast(value)
    except (TypeError, ValueError):
        return default


def normalize_record(raw, doc_id=None):
    """Turn a raw dict into a clean record with all six fields.
    Raises ValueError if the record has no usable text or no doc_id."""
    if not isinstance(raw, dict):
        raise ValueError("record is not a dict")

    # text is the only truly required field
    text = raw.get('text')
    text = text.strip() if isinstance(text, str) else ''
    if not text:
        raise ValueError("record has no text")

    title = raw.get('title')
    title = title.strip() if isinstance(title, str) else ''

    # doc_id: use the one passed in, otherwise the one in the record
    if doc_id is None:
        doc_id = _to_number(raw.get('doc_id'), int, None)
    if doc_id is None:
        raise ValueError("record has no doc_id")

    rating = _to_number(raw.get('rating'), float, DEFAULT_RATING)
    rating = min(5.0, max(1.0, rating))            # keep inside 1 to 5

    votes = _to_number(raw.get('helpful_vote'), int, 0)
    votes = max(0, votes)                           # never negative

    now_ms = int(time.time() * 1000)                # Amazon timestamps are in milliseconds
    timestamp = _to_number(raw.get('timestamp'), int, now_ms)

    return {
        "doc_id": doc_id,
        "title": title,
        "text": text,
        "rating": rating,
        "helpful_vote": votes,
        "timestamp": timestamp,
    }


# --- Tests: broken records must be fixed or rejected, never crash ---
print(normalize_record({"text": "Great game", "rating": None,
                        "helpful_vote": None, "timestamp": None}, doc_id=1))
print(normalize_record({"text": "Fine", "rating": "9", "helpful_vote": -4}, doc_id=2))

try:
    normalize_record({"text": "   "}, doc_id=3)
except ValueError as e:
    print("Rejected empty review:", e)

{'doc_id': 1, 'title': '', 'text': 'Great game', 'rating': 3.0, 'helpful_vote': 0, 'timestamp': 1795267543728}
{'doc_id': 2, 'title': '', 'text': 'Fine', 'rating': 5.0, 'helpful_vote': 0, 'timestamp': 1795267543728}
Rejected empty review: record has no text


In [10]:
#
# Cell 2B: analyze_document + iter_records
#
def analyze_document(record):
    """Return (stats, doc_len) for one clean record.
    stats maps each term to [tf, in_title, first_pos]:
      tf        = how many times the term appears in the text
      in_title  = 1 if the term also appears in the title, else 0
      first_pos = index of the first occurrence in the text (-1 if title-only)
    doc_len = number of tokens in the text."""
    text_tokens = tokenize(record['text'])

    stats = {}
    for pos, term in enumerate(text_tokens):
        if term in stats:
            stats[term][0] += 1              # seen before: just add to tf
        else:
            stats[term] = [1, 0, pos]        # first time: tf=1, remember position

    # Title terms: use a set so a repeated title word is counted once
    for term in set(tokenize(record['title'])):
        if term in stats:
            stats[term][1] = 1               # word is in text and in title
        else:
            stats[term] = [0, 1, -1]         # title-only word

    return stats, len(text_tokens)


def iter_records(path):
    """Yield clean records from a JSONL file, one at a time.
    Bad lines (invalid JSON, no text, no doc_id) are skipped and counted."""
    skipped = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue                     # ignore blank lines
            try:
                yield normalize_record(json.loads(line))
            except ValueError as e:          # bad JSON is also a ValueError
                skipped += 1
                if skipped <= 5:
                    print(f"Skipped line {line_no}: {e}")
    if skipped:
        print(f"Total skipped lines: {skipped}")


# --- Test 1: analyze one small document by hand ---
stats, n = analyze_document({"title": "Great battery",
                             "text": "The battery lasts long. Battery good."})
print("doc_len:", n)
print(stats)

# --- Test 2: stream the sample file ---
count = 0
for rec in iter_records(SOURCE_PATH):
    count += 1
print("Records read from source:", count)

doc_len: 6
{'the': [1, 0, 0], 'batteri': [2, 1, 1], 'last': [1, 0, 2], 'long': [1, 0, 3], 'good': [1, 0, 5], 'great': [0, 1, -1]}
Records read from source: 1000


In [11]:

# Cell 2C: Lexicon  ->  lexicon.json  {word: {"id": n, "df": n}}

def build_lexicon(source_path, lexicon_path): # func that takes 2 things: srs and dest(lexicon)

    """Read every review, count in how many reviews each word appears (df),
    give each unique word an ID, and save the lexicon as a JSON file."""

    # df: document freq.  is a dictionary that will map each word to the number of reviews that contain it.
    df = {}                                    # word -> number of reviews containing it

    for record in iter_records(source_path):
        stats, _ = analyze_document(record)    # stats has one key per unique word
        for term in stats:
            df[term] = df.get(term, 0) + 1     # this review contains the word once

    # Sorted order gives the same IDs on every run (reproducible)
    lexicon = {term: {"id": i, "df": df[term]}
               for i, term in enumerate(sorted(df))}

    with open(lexicon_path, 'w', encoding='utf-8') as f:
        json.dump(lexicon, f)
    return lexicon


t0 = time.time()
lexicon = build_lexicon(SOURCE_PATH, LEXICON_PATH)
elapsed = time.time() - t0

print(f"Unique words in lexicon: {len(lexicon):,}  (built in {elapsed:.1f}s)")
print("First 3 entries:", {w: lexicon[w] for w in list(lexicon)[:3]})

# Most common words by document frequency
top = sorted(lexicon.items(), key=lambda kv: -kv[1]["df"])[:8]
print("Most common:", [(w, info["df"]) for w, info in top])

# Check: IDs must be exactly 0 .. n-1 with no gaps or repeats
ids = sorted(info["id"] for info in lexicon.values())
print("IDs are 0..n-1 with no gaps:", ids == list(range(len(lexicon))))
print(f"Saved at: {LEXICON_PATH} ({os.path.getsize(LEXICON_PATH)/1e6:.2f} MB)")

Unique words in lexicon: 9,245  (built in 0.9s)
First 3 entries: {'0': {'id': 0, 'df': 19}, '00': {'id': 1, 'df': 4}, '000': {'id': 2, 'df': 5}}
Most common: [('the', 998), ('and', 983), ('a', 981), ('to', 978), ('i', 942), ('it', 939), ('is', 927), ('thi', 909)]
IDs are 0..n-1 with no gaps: True
Saved at: /content/drive/MyDrive/DSA_Search_Engine/index_data/sample/lexicon.json (0.30 MB)
